# Convective storm detection and tracking

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. Extract a temperature loop

In [ ]:
import mcidasv_jupyter as mcv
import numpy as np
session = mcv.get_session()
N = 5

fields = session.extract_fields('''
adde = dict(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
            size='ALL', unit='TEMP', mag=(-8, -8))
frames = [loadADDEImage(position=p, **adde) for p in range(-(N-1), 1)]
panel = buildWindow(height=400, width=500)
layer = panel[0].createLayer('Image Sequence Display', frames)
''', values={'N': N}, times=range(N))

cube = np.stack([f.masked() for f in fields])
nav = fields[0]
print(cube.shape, nav.unit)

## 2. Detect cold cores (Tb < 220 K)

In [ ]:
from scipy import ndimage as ndi

THRESH_K = 220.0
MIN_PIX = 20

def detect(tb):
    mask = np.isfinite(tb) & (tb < THRESH_K)
    lbl, n = ndi.label(mask)
    cells = []
    for i in range(1, n + 1):
        sel = lbl == i
        if sel.sum() < MIN_PIX:
            continue
        cells.append(dict(lat=float(nav.lats[sel].mean()),
                          lon=float(nav.lons[sel].mean()),
                          npix=int(sel.sum()),
                          tmin=float(np.nanmin(tb[sel]))))
    return cells

detections = [detect(tb) for tb in cube]
print('cells per frame:', [len(d) for d in detections])

## 3. Track cells in geographic space

In [ ]:
GATE_KM = 120.0

def km_between(a, b):
    dlat = (a['lat'] - b['lat']) * 111.0
    dlon = (a['lon'] - b['lon']) * 111.0 * np.cos(np.radians(a['lat']))
    return float(np.hypot(dlat, dlon))

tracks, active = [], []
for fi, cells in enumerate(detections):
    used = set()
    for tr in active:
        _, last = tr[-1]
        best, bestd = None, GATE_KM
        for j, c in enumerate(cells):
            if j in used: continue
            d = km_between(c, last)
            if d < bestd: best, bestd = j, d
        if best is not None:
            tr.append((fi, cells[best])); used.add(best)
    for j, c in enumerate(cells):
        if j not in used:
            tracks.append([(fi, c)])
    active = [tr for tr in tracks if tr[-1][0] == fi]

long_tracks = [t for t in tracks if len(t) >= 3]
print('tracks:', len(tracks), '| >=3 frames:', len(long_tracks))
for tr in long_tracks[:5]:
    a, b = tr[0][1], tr[-1][1]
    print('  %.2fN %.2fE -> %.2fN %.2fE | %d -> %d px | Tmin %.1f -> %.1f K' % (
        a['lat'], a['lon'], b['lat'], b['lon'], a['npix'], b['npix'], a['tmin'], b['tmin']))

## 4. Plot tracks and cell growth

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].imshow(cube[-1], cmap='inferno_r'); ax[0].set_title('Tb (K), last frame'); ax[0].axis('off')
for tr in long_tracks:
    lats = [c['lat'] for _, c in tr]; lons = [c['lon'] for _, c in tr]
    ax[1].plot(lons, lats, '-o', ms=4)
ax[1].set_xlabel('lon'); ax[1].set_ylabel('lat'); ax[1].set_title('tracks (geographic)')
ax[1].grid(alpha=.3)
plt.tight_layout()

## 5. Mark the detected cells in McIDAS-V

In [ ]:
last = detections[-1]
ann = ["panel[0].annotate('X', lat=%f, lon=%f, size=18, color='red')" % (c['lat'], c['lon'])
       for c in last]

base = '''
data = loadADDEImage(server='adde.ucar.edu', dataset='EAST', descriptor='CONUSC13',
                     size='ALL', unit='TEMP')
panel = buildWindow(height=500, width=750)
layer = panel[0].createLayer('Image Display', data)
panel[0].setProjection('US>CONUS')
panel[0].setWireframe(False)
layer.setEnhancement('ABI IR Temperature', range=(200, 300))
'''
session.run(base + '\n'.join(ann))